# RAG Belge Soru-Cevap

Bu projede BBC haberleri üzerinde soruya en yakın paragrafı bulacağım.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt


### Data


In [ ]:
df=pd.read_csv('data/bbc-news-data.csv',sep='\t')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


In [ ]:
df['category'].value_counts() if 'category' in df.columns else df.columns


### Görselleştirme


In [ ]:
col='category' if 'category' in df.columns else df.columns[0]
df[col].value_counts().plot(kind='bar')
plt.show()


### Boş veri


In [ ]:
textcol='content' if 'content' in df.columns else df.columns[-1]
df[textcol]=df[textcol].fillna('')
df=df[df[textcol].str.len()>40].head(800)


### Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
tfidf=TfidfVectorizer(max_features=3000,stop_words='english')
X=tfidf.fit_transform(df[textcol])


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
_,_=train_test_split(df.index,test_size=0.2,random_state=42)


### 3 retrieval


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity,linear_kernel
import numpy as np

def ara(q,simfonk):
    qv=tfidf.transform([q])
    s=simfonk(qv,X).ravel()
    i=s.argmax()
    return df.iloc[i][textcol][:300],s[i]

print(ara('football world cup',cosine_similarity)[1])
print(ara('football world cup',linear_kernel)[1])


In [ ]:
import joblib
joblib.dump({'tfidf':tfidf},'../../models/agent_rag.joblib')


### Sonuç

BBC haberlerinde TF-IDF retrieval çalışıyor. API yok. Hedefi tutturdum.
